# FactLedger extractor

`load(path) -> documents, units` over raw files and nothing else. The extractor sniffs the
format from the bytes and writes document and unit JSON in the shapes of SCHEMA.md; the
rules it follows are in BUILD.md. Built one block at a time. Inputs: the public raw dataset
and the private papers dataset, both attached to this notebook.


In [ ]:
# Block 1: inputs and integrity.
# Mount both datasets, count files per folder, and check every file's sha256 against the
# folder manifest. The manifests are used here only to prove the Kaggle copies are the bytes
# that were uploaded; the extractor itself never reads them.
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def mount(slug):
    """Kaggle mounts inputs at /kaggle/input/<slug> or, in newer sessions,
    /kaggle/input/datasets/<owner>/<slug>. Take whichever exists."""
    for candidate in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets/jhffmn") / slug):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(slug)


RAW = mount("it494-narrative-corpora-raw")
PAPERS = mount("it494-reference-papers")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def check(folder, rows_key):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    rows = manifest[rows_key]
    on_disk = {p.name for p in folder.iterdir() if p.name not in ("manifest.json", "LICENSE")}
    listed = {r["file"] for r in rows}
    # Kaggle inputs are a network filesystem: one file at a time, 19,206 files take tens of
    # minutes; 32 concurrent reads take about a minute.
    with ThreadPoolExecutor(max_workers=32) as pool:
        digests = list(pool.map(sha256, [folder / r["file"] for r in rows]))
    bad = [r["file"] for r, d in zip(rows, digests) if d != r["sha256"]]
    print(f"{folder.name:<28} files {len(on_disk):>6}  listed {len(listed):>6}"
          f"  mismatched {len(bad)}  unlisted {len(on_disk - listed)}  missing {len(listed - on_disk)}")
    return bad


# The three literature manifests keep their original "works" key; the unpacked folders
# and the papers use "files".
for name, key in [("oz", "works"), ("holmes", "works"), ("greek", "works"),
                  ("graphrag-bench", "files"), ("longmemeval", "files")]:
    check(RAW / name, key)
check(PAPERS, "files")


In [ ]:
# Block 2: file type, then raw text.
#
# Two steps, by bytes only. Nothing here decides what the text is about; that is the model's
# job later.
#   1. file_kind(data): look at the first bytes and name the container: pdf, json, or text.
#   2. to_text(path): turn the container into one string, the document text.
#        pdf  -> the text layer, page by page (PyMuPDF, the one dependency)
#        json -> if it holds chat turns, one "role: content" block per turn under a header
#                of the session id and dates. We also keep where each turn starts and ends
#                in that string, so a chat can be cut into units without a model.
#        text -> the bytes decoded as UTF-8, unchanged
import json

try:
    import pymupdf
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import pymupdf


def file_kind(data):
    if data.startswith(b"%PDF-"):
        return "pdf"
    if data.lstrip()[:1] in (b"{", b"["):
        return "json"
    return "text"


def pdf_text(data):
    doc = pymupdf.open(stream=data, filetype="pdf")
    return "\n".join(page.get_text() for page in doc)


def chat_turns(obj):
    """The list of {role, content} turns inside a chat JSON, wherever it sits; None if absent."""
    if isinstance(obj, list) and obj and all(isinstance(t, dict) and "role" in t and "content" in t for t in obj):
        return obj
    if isinstance(obj, dict):
        for value in obj.values():
            found = chat_turns(value)
            if found:
                return found
    return None


def chat_text(obj, turns):
    """Header lines, a blank line, then 'role: content' per turn. Returns the text and the
    (start, end) of each turn inside it."""
    header = [f"session_id: {obj['session_id']}"] if "session_id" in obj else []
    header += [f"date: {d}" for d in obj.get("dates", [])]
    text = "\n".join(header) + "\n\n"
    spans = []
    for turn in turns:
        start = len(text)
        text += f"{turn['role']}: {turn['content']}\n\n"
        spans.append((start, len(text)))
    return text, spans


def to_text(path):
    data = path.read_bytes()
    doc = {"path": str(path), "sha256": hashlib.sha256(data).hexdigest(),
           "kind": file_kind(data), "text": "", "turns": None, "dates": []}
    if doc["kind"] == "pdf":
        doc["text"] = pdf_text(data)
    elif doc["kind"] == "json":
        obj = json.loads(data)
        turns = chat_turns(obj)
        if turns is None:
            doc["kind"], doc["text"] = "text", data.decode("utf-8", errors="replace")
        else:
            doc["kind"] = "chat"
            doc["text"], doc["turns"] = chat_text(obj, turns)
            doc["dates"] = list(obj.get("dates", []))
    else:
        doc["text"] = data.decode("utf-8", errors="replace")
    return doc


# One of each, to see the shape.
for path in [RAW / "oz" / "01_55.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
             RAW / "longmemeval" / "sharegpt_yywfIrx_0.json", RAW / "longmemeval" / "001cefa7_2.json",
             PAPERS / "edge2024-graphrag.pdf"]:
    d = to_text(path)
    turns = len(d["turns"]) if d["turns"] else "-"
    print(f"{path.name:<26} {d['kind']:<5} {len(d['text']):>8,} chars  turns {turns:>3}  dates {d['dates']}")
    print("    " + repr(d["text"][:70]))


In [ ]:
# Block 3: the model interface.
# One JSON-mode call at a time, raw HTTP, every call logged with model, tokens, seconds,
# and cost. The spending stop is checked before each call. The matcher is the gate: a
# phrase the model returns must occur in the text, line wraps and spacing forgiven, and
# `locate` says where, searching forward from a position so a table of contents cannot
# steal a heading that appears again in the body.
import os
import re
import time

import requests

try:
    from kaggle_secrets import UserSecretsClient
    KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
except ImportError:
    KEY = os.environ.get("OPENAI_API_KEY", "")

MODEL = "gpt-5.6-luna"                # the split call
RETRY = "gpt-5.6-terra"               # one retry when the gates fail
PRICE = {"gpt-5.6-luna": (0.20, 1.20), "gpt-5.6-terra": (2.00, 12.00)}   # $ per M tokens in, out
SPEND_STOP = 2.00                     # dollars for the whole raw dataset; the run halts past this
calls = []


def spend():
    return sum(c["cost"] for c in calls)


def generate(prompt, stage, model=MODEL, effort="low"):
    """One JSON-mode call; the reply parsed, the cost logged. No temperature: the API
    rejects it; reasoning_effort is the only knob."""
    if spend() >= SPEND_STOP:
        raise RuntimeError(f"spending stop: ${spend():.2f}")
    t0 = time.time()
    r = requests.post("https://api.openai.com/v1/chat/completions",
                      headers={"Authorization": f"Bearer {KEY}"}, timeout=300,
                      json={"model": model, "reasoning_effort": effort,
                            "response_format": {"type": "json_object"},
                            "messages": [{"role": "user", "content": prompt}]})
    if r.status_code != 200:
        raise RuntimeError(f"OpenAI {r.status_code}: {r.text}")
    body = r.json()
    u, (p_in, p_out) = body["usage"], PRICE[model]
    calls.append({"stage": stage, "model": body["model"], "in": u["prompt_tokens"],
                  "out": u["completion_tokens"], "seconds": round(time.time() - t0, 1),
                  "cost": (u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out) / 1e6})
    return json.loads(body["choices"][0]["message"]["content"])


def pattern(s):
    return r"\s+".join(re.escape(w) for w in s.split())


def in_text(text, s):
    """The gate: the phrase occurs in the text, line wraps forgiven."""
    return bool(s.split()) and re.search(pattern(s), text) is not None


def locate(text, s, start=0):
    """(start, end) of the first occurrence of the phrase at or after `start`, else None."""
    if not s or not s.split():
        return None
    m = re.compile(pattern(s)).search(text, start)
    return (m.start(), m.end()) if m else None


print(f"model {MODEL}, retry {RETRY}, spend stop ${SPEND_STOP:.2f}, key {'present' if KEY else 'MISSING'}")


In [ ]:
# Block 4: the compressed view.
#
# The model never sees a whole book. It sees a numbered list of the places structure lives:
#   line view   the first and last 200 lines in full (title page, contents, the ending), and
#               in between only the short non-empty lines, because headings are short. "Short"
#               starts at 60 characters and tightens until the middle fits 3,000 entries, so a
#               verse play (every line short) still fits.
#   sentence view   for a text with almost no line breaks (one-line files), every sentence
#               start, first 60 characters each. A heading glued to the sentence after it
#               shows up as the start of that sentence.
# Every entry is verbatim text, so whatever the model quotes back can be found again.
import re

EDGE = 200
MAX_MIDDLE = 3000
PREFIX = 60
SENTENCE_START = re.compile(r'(?<=[.!?])\s+(?=[A-Z"“(\[])')


def compressed_view(text):
    if text.count("\n") < len(text) / 500:
        sentences = SENTENCE_START.split(text)
        entries = [(i, s[:PREFIX]) for i, s in enumerate(sentences)]
        mode, limit = "sentences", None
    else:
        lines = text.split("\n")
        first, last = range(min(EDGE, len(lines))), range(max(EDGE, len(lines) - EDGE), len(lines))
        for limit in (60, 40, 30, 20, 12):
            middle = [i for i in range(EDGE, len(lines) - EDGE) if lines[i].strip() and len(lines[i].rstrip("\r")) <= limit]
            if len(middle) <= MAX_MIDDLE:
                break
        entries = [(i, lines[i].rstrip("\r")[:120]) for i in [*first, *middle, *last]]
        mode = "lines"
    step = max(1, -(-len(entries) // (MAX_MIDDLE + 2 * EDGE)))    # ceiling division
    entries = entries[::step]
    body = "\n".join(f"{i}: {s}" for i, s in entries)
    return {"mode": mode, "limit": limit, "entries": len(entries), "sampled_every": step,
            "chars": len(body), "text": body}


for path in [RAW / "oz" / "01_55.txt", RAW / "greek" / "18_10523.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
             PAPERS / "edge2024-graphrag.pdf"]:
    v = compressed_view(to_text(path)["text"])
    lines = v["text"].split("\n")
    print(f"{path.name:<26} {v['mode']:<9} short<={v['limit']}  entries {v['entries']:>5}  ~tokens {v['chars'] // 4:>6}"
          f"  sampled every {v['sampled_every']}")
    print("    " + "\n    ".join(lines[len(lines) // 2: len(lines) // 2 + 6]))


In [ ]:
# Block 5: the split call.
#
# One call per document. The model reads the compressed view and answers with JSON: what the
# document is, its title, author, translator, and date, where the work itself starts and
# ends (so front matter and end matter fall away), and the heading that opens each piece,
# with a short title for the piece. Every string it returns must be copied verbatim from the
# view, because the next block searches the document for it and cuts there. Nothing the
# model says is trusted until that search finds it.
CAP_WORDS = 4000

PROMPT = """Below is a compressed view of one document: numbered lines, or numbered sentence starts when the document has no line breaks. In the line view the first and last 200 lines are complete and the middle shows only short lines, which is where headings are.

Decide what the document is and how it divides into pieces. Answer with JSON only. Every string you return must be copied exactly from the view, character for character, never paraphrased or corrected, because a program will search the document for it.

{
  "kind": one of "novel", "play", "poetry", "anthology", "story collection", "paper", "article", "essay", "notes", "email", "letter", "diary", "reference", "other",
  "evidence": a verbatim phrase from the view that shows the kind,
  "title": the title as written, or null,
  "author": the author's name as written, or null,
  "translator": the translator's name as written, or null,
  "date": {"quote": a verbatim phrase that contains the date, "iso": "YYYY" or "YYYY-MM" or "YYYY-MM-DD", "meaning": one of "publication", "written", "sent", "release", "other"} or null,
  "body_start": the verbatim opening of the first line of the work itself, after any front matter (publisher notices, contents, preface, translator's introduction),
  "body_end": the verbatim opening of the last line of the work itself, before any end matter (license, index, notes, advertisements),
  "toc_count": the number of pieces a table of contents lists, or null if there is none,
  "pieces": [{"marker": the verbatim opening of the heading line that begins the piece, in document order, "title": a short title for the piece, such as "Chapter 1: The Cyclone" or "Abstract" or "Act II, Scene 1"}]
}

Pieces are the document's own divisions: chapters, acts and scenes, sections, dated entries, poems, stories. A paper's pieces are its sections, the abstract first. Aim for pieces under %d words; where a division is longer, use its next level down. A document with no divisions gets an empty pieces list. Front matter and end matter are never pieces. Do not take markers from a table of contents; markers are the headings where each piece begins in the body.

VIEW:
%s
"""


def propose(doc, model=MODEL):
    """The model's proposal for one document, plus the view it saw."""
    view = compressed_view(doc["text"])
    reply = generate(PROMPT % (CAP_WORDS, view["text"]), stage="split", model=model)
    return reply, view


doc = to_text(RAW / "oz" / "01_55.txt")
reply, view = propose(doc)
for key in ("kind", "evidence", "title", "author", "translator", "date", "body_start", "body_end", "toc_count"):
    print(f"{key:<11} {reply.get(key)!r}")
print(f"pieces     {len(reply.get('pieces', []))}")
for piece in reply.get("pieces", [])[:5]:
    print(f"    {piece}")
print(f"call: {calls[-1]}")
